In [50]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import RobustScaler
from sktime.forecasting.chronos import ChronosForecaster
from sktime.forecasting.conformal import ConformalIntervals

from sktime.forecasting.model_selection import ForecastingGridSearchCV
from sktime.forecasting.compose import make_reduction, TransformedTargetForecaster, ForecastingPipeline
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.arima import AutoARIMA
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError
from sktime.performance_metrics.forecasting.probabilistic import PinballLoss
from sktime.split import ExpandingWindowSplitter, temporal_train_test_split
from sktime.transformations.compose import OptionalPassthrough
from sktime.transformations.series.adapt import TabularToSeriesAdaptor
from sktime.transformations.series.detrend import Detrender, Deseasonalizer
from sktime.transformations.series.impute import Imputer

# Data Prep

In [3]:
data = (pl.read_csv('../../data/processed/final_feature_df.csv',
                   try_parse_dates=True,
                   columns=['date', 'sales', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'year_sin', 'year_cos', 'is_lecture_day', 'is_regular_service_day', 'sunshine_duration', 'apparent_temperature_mean',],
                   )
        .filter(pl.col('is_regular_service_day'))
        .drop('is_regular_service_day')
        .with_columns(pl.col('is_lecture_day').cast(pl.Float64))
        )
data

date,week_sin,week_cos,month_sin,month_cos,year_sin,year_cos,is_lecture_day,sunshine_duration,apparent_temperature_mean,sales
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
2025-01-06,0.0,1.0,0.858764,0.512371,0.085965,0.996298,1.0,4.9225845,-1.277526,1866
2025-01-07,0.781831,0.62349,0.945596,0.325342,0.103102,0.994671,1.0,0.0,-0.479467,2224
2025-01-08,0.974928,-0.222521,0.992222,0.124479,0.120208,0.992749,1.0,0.1398955,1.6230611,2153
2025-01-09,0.433884,-0.900969,0.996659,-0.081676,0.137279,0.990532,1.0,4.0080094,-3.397642,1739
2025-01-10,-0.433884,-0.900969,0.958718,-0.284359,0.154309,0.988023,1.0,0.4939937,-3.33542,1366
…,…,…,…,…,…,…,…,…,…,…
2025-12-15,0.0,1.0,0.361714,-0.932289,-0.288482,0.957485,1.0,6.0202994,3.3126433,1408
2025-12-16,0.781831,0.62349,0.162807,-0.986658,-0.271958,0.962309,1.0,4.8902845,2.2815366,1483
2025-12-17,0.974928,-0.222521,-0.043022,-0.999074,-0.255353,0.966848,1.0,4.644257,5.767552,1459


In [4]:
data_pd = data.to_pandas().set_index("date")
data_pd["original_date"] = data_pd.index

data_pd.index = pd.period_range(
    start=data_pd.index.min(),
    periods=len(data_pd),
    freq="B",
)

y = data_pd["sales"]
X = data_pd.drop(columns=["sales", "original_date"])

C:\Users\valen\AppData\Local\Temp\ipykernel_5488\1986072616.py:4: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  data_pd.index = pd.period_range(
C:\Users\valen\AppData\Local\Temp\ipykernel_5488\1986072616.py:4: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  data_pd.index = pd.period_range(


In [5]:
y = y.interpolate().ffill().bfill()
X = X.ffill().bfill()

# Pipeline Setup

## Preprocessors

In [6]:
pipe_y = TransformedTargetForecaster(
    steps=[
        ("y_imputer", Imputer(method="constant", value=0)),
        ("detrender", OptionalPassthrough(Detrender())),
        ("deseasonalizer", OptionalPassthrough(Deseasonalizer())),
        ("scaler", TabularToSeriesAdaptor(RobustScaler())),
        ("forecaster", NaiveForecaster(strategy="mean", sp=5)),
    ]
)

In [7]:
pipe = ForecastingPipeline(
    steps=[
        ("x_scaler", TabularToSeriesAdaptor(RobustScaler())),
        ("forecaster", pipe_y),
    ]
)

## Params

In [32]:
proba_forecaster_param_grid = [
    # Naive baseline
    {
        "detrender__passthrough": [True, False],
        "deseasonalizer__passthrough": [True, False],
        "forecaster": [
            NaiveForecaster(strategy="last", sp=5),
            NaiveForecaster(strategy="mean", sp=5),
        ],
    },

    # SARIMAX
    {
        "detrender__passthrough": [True],
        "deseasonalizer__passthrough": [True],
        "forecaster": [
            AutoARIMA(sp=5, seasonal=True, stationary=False, suppress_warnings=True),
            AutoARIMA(sp=5, seasonal=False, stationary=False, suppress_warnings=True),
        ],
        "forecaster__max_p": [2, 4],
        "forecaster__max_q": [2, 4],
        "forecaster__max_P": [1, 2],
        "forecaster__max_Q": [1, 2],
    },
]

point_forecaster_param_grid = [
    # Gradient boosting
    {
        "forecaster__detrender__passthrough": [True, False],
        "forecaster__deseasonalizer__passthrough": [True, False],
        "forecaster__forecaster": [
            make_reduction(
                HistGradientBoostingRegressor(loss="squared_error", random_state=42),
                strategy="recursive",
                window_length=5,
            )
        ],
        "forecaster__forecaster__window_length": [5, 10, 20],
        "forecaster__forecaster__estimator__max_iter": [100, 300],
        "forecaster__forecaster__estimator__learning_rate": [0.01, 0.1],
        "forecaster__forecaster__estimator__max_leaf_nodes": [15, 31],
        "forecaster__forecaster__estimator__l2_regularization": [0.0, 0.1],
    },

    # Quantile regression
    {
        "forecaster__detrender__passthrough": [True, False],
        "forecaster__deseasonalizer__passthrough": [True, False],
        "forecaster__forecaster": [
            make_reduction(
                QuantileRegressor(quantile=0.5, solver="highs"),
                strategy="recursive",
                window_length=5,
            )
        ],
        "forecaster__forecaster__window_length": [5, 10, 20],
        "forecaster__forecaster__estimator__alpha": [0.01, 0.1, 1.0],
    },

    # KNN
    {
        "forecaster__detrender__passthrough": [True, False],
        "forecaster__deseasonalizer__passthrough": [True, False],
        "forecaster__forecaster": [
            make_reduction(
                KNeighborsRegressor(weights='distance', n_jobs=-1), # weight by distance for a stronger effect of close neighbors
                strategy="recursive",
                window_length=5,
            )
        ],
        "forecaster__forecaster__window_length": [5, 10, 20],
        "forecaster__forecaster__estimator__n_neighbors": [3, 5, 10],
        "forecaster__forecaster__estimator__p": [1, 2],
    },

    # Chronos Foundation Model, no detrending and deseasonalization is done here because the model can handle this type of data
    {
        "forecaster__detrender__passthrough": [True],
        "forecaster__deseasonalizer__passthrough": [True],
        "forecaster__forecaster": [
            ChronosForecaster(
                model_path="amazon/chronos-t5-base",
                seed=42,
            ),
        ],
    },
]

In [33]:
fh = [1,2,3,4,5]

# expanding window splitter with 4 weeks á 5 days initial window length and 5 working day stride, only works because the data starts at monday and excludes weekends, problems may arise when holidays occur
cv = ExpandingWindowSplitter(fh=fh, initial_window=30, step_length=5)
proba_scoring = PinballLoss(alpha=[0.1, 0.25, 0.5, 0.75, 0.9])
point_scoring = MeanAbsolutePercentageError()

## Grid Search

In [34]:
y_train, y_test, X_train, X_test = temporal_train_test_split(y, X, fh=fh)

In [35]:
# for the two probabilistic models we do not use exogeneous variables for now, the naive case doesnt care about X and ARIMA does not work with exogeneous at the moment -> TODO: fix later
proba_gscv = ForecastingGridSearchCV(pipe_y, cv=cv, param_grid=proba_forecaster_param_grid, scoring=proba_scoring, error_score="raise")
point_gscv = ForecastingGridSearchCV(pipe, cv=cv, param_grid=point_forecaster_param_grid, scoring=point_scoring, error_score="raise")

In [41]:
proba_gscv.fit(y_train, fh=fh)
proba_gscv.best_params_

{'deseasonalizer__passthrough': True,
 'detrender__passthrough': True,
 'forecaster': AutoARIMA(max_P=1, max_Q=1, max_p=2, max_q=4, sp=5, suppress_warnings=True),
 'forecaster__max_P': 1,
 'forecaster__max_Q': 1,
 'forecaster__max_p': 2,
 'forecaster__max_q': 4}

In [36]:
point_gscv.fit(y_train, X=X_train, fh=fh)
point_gscv.best_params_

{'forecaster__deseasonalizer__passthrough': True,
 'forecaster__detrender__passthrough': True,
 'forecaster__forecaster': RecursiveTabularRegressionForecaster(estimator=KNeighborsRegressor(n_jobs=-1, n_neighbors=3, p=1, weights='distance'),
                                      window_length=5),
 'forecaster__forecaster__estimator__n_neighbors': 3,
 'forecaster__forecaster__estimator__p': 1,
 'forecaster__forecaster__window_length': 5}

### Find best point forecaster and transform it to interval forecaster

In [53]:
best_point_forecaster = point_gscv.best_forecaster_.clone()

conformal_point_forecaster = ConformalIntervals(
    forecaster=best_point_forecaster,
    method="empirical",
    initial_window=30,
    n_jobs=-1
)

conformal_point_forecaster.fit(
    y=y_train,
    X=X_train,
    fh=fh,
)

C:\Users\valen\AppData\Local\Programs\Python\Python312\Lib\multiprocessing\connection.py:251: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return _ForkingPickler.loads(buf.getbuffer())


ConformalIntervals(forecaster=ForecastingPipeline(steps=[('x_scaler',
                                                          TabularToSeriesAdaptor(transformer=RobustScaler())),
                                                         ('forecaster',
                                                          TransformedTargetForecaster(steps=[('y_imputer',
                                                                                              Imputer(method='constant',
                                                                                                      value=0)),
                                                                                             ('detrender',
                                                                                              OptionalPassthrough(passthrough=True,
                                                                                                                  transformer=Detrender())),
                                                                                             ('deseasonalizer',
                                                                                              OptionalPassthrough(passthrough=True,
                                                                                                                  transformer=Deseasonalizer())),
                                                                                             ('scaler',
                                                                                              TabularToSeriesAdaptor(transformer=RobustScaler())),
                                                                                             ('forecaster',
                                                                                              RecursiveTabularRegressionForecaster(estimator=KNeighborsRegressor(n_jobs=-1, n_neighbors=3, p=1, weights='distance'),
                                                                                                                                   window_length=5))]))]),
                   initial_window=30, n_jobs=-1)

### Evaluate on test set

In [54]:
coverage = [0.75, 0.9]

proba_interval = proba_gscv.best_forecaster_.predict_interval(
    fh=fh,
    X=X_test,
    coverage=coverage,
)

point_conformal_interval = conformal_point_forecaster.predict_interval(
    fh=fh,
    X=X_test,
    coverage=coverage,
)

C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:140: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  index = index_fn(
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:140: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  index = index_fn(
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:140: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  index = index_fn(
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:140: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  inde

In [55]:
interval_pinball_by_coverage = PinballLoss(score_average=False)

proba_interval_score_by_coverage = interval_pinball_by_coverage.evaluate(
    y_true=y_test,
    y_pred=proba_interval,
)

point_conformal_interval_score_by_coverage = interval_pinball_by_coverage.evaluate(
    y_true=y_test,
    y_pred=point_conformal_interval,
)

In [56]:
interval_pinball = PinballLoss()

scores = {
    "proba_gscv": interval_pinball.evaluate(y_test, proba_interval),
    "point_conformal": interval_pinball.evaluate(y_test, point_conformal_interval),
}

scores

{'proba_gscv': np.float64(37.75038161189758),
 'point_conformal': 41.554207833012484}

# Full Eval - Vibe Code Slop

In [ ]:
coverage = [0.75, 0.90]

interval_pinball = PinballLoss()
interval_pinball_by_coverage = PinballLoss(score_average=False)
point_mape = MeanAbsolutePercentageError()


def _as_series(y):
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError("Plotting output currently expects univariate y.")
        return y.iloc[:, 0]
    return y


def _interval_column(interval_df, cov, bound):
    for col in interval_df.columns:
        if not isinstance(col, tuple):
            continue

        has_bound = bound in col
        has_coverage = any(
            isinstance(x, (float, int, np.floating, np.integer))
            and np.isclose(float(x), float(cov))
            for x in col
        )

        if has_bound and has_coverage:
            return col

    raise KeyError(f"Could not find {bound} interval column for coverage={cov}")


def _model_label(prefix, i, params):
    if not params:
        return f"{prefix}_{i:03d}"

    readable_params = ", ".join(f"{k}={v}" for k, v in params.items())
    return f"{prefix}_{i:03d}: {readable_params}"


y_test_series = _as_series(y_test)

all_interval_predictions = {}
score_rows = []
plot_frames = []


# -------------------------------------------------------------------
# 1. Probabilistic grid-search candidates
# -------------------------------------------------------------------

for i, params in enumerate(proba_gscv.cv_results_["params"]):
    model_name = _model_label("proba", i, params)

    forecaster = proba_gscv.forecaster.clone()
    forecaster.set_params(**params)

    # Same setup as your grid search: no exogenous X for probabilistic models
    forecaster.fit(y_train, fh=fh)

    interval_pred = forecaster.predict_interval(
        fh=fh,
        coverage=coverage,
    )

    all_interval_predictions[model_name] = interval_pred

    try:
        point_pred = _as_series(forecaster.predict(fh=fh))
    except Exception:
        point_pred = pd.Series(index=y_test_series.index, dtype=float)

    overall_pinball = interval_pinball.evaluate(
        y_true=y_test,
        y_pred=interval_pred,
    )

    pinball_by_coverage = interval_pinball_by_coverage.evaluate(
        y_true=y_test,
        y_pred=interval_pred,
    )

    try:
        mape = point_mape.evaluate(
            y_true=y_test,
            y_pred=point_pred,
        )
    except Exception:
        mape = np.nan

    for cov in coverage:
        lower = interval_pred[_interval_column(interval_pred, cov, "lower")]
        upper = interval_pred[_interval_column(interval_pred, cov, "upper")]

        score_rows.append(
            {
                "model": model_name,
                "model_family": "probabilistic",
                "coverage": cov,
                "test_pinball_loss": overall_pinball,
                "test_pinball_loss_by_coverage": (
                    pinball_by_coverage.loc[cov]
                    if cov in getattr(pinball_by_coverage, "index", [])
                    else np.nan
                ),
                "test_mape": mape,
                "observed_coverage": (
                    (y_test_series >= lower) & (y_test_series <= upper)
                ).mean(),
                "average_interval_width": (upper - lower).mean(),
            }
        )

        plot_frames.append(
            pd.DataFrame(
                {
                    "timestamp": y_test_series.index,
                    "model": model_name,
                    "model_family": "probabilistic",
                    "coverage": cov,
                    "y_true": y_test_series.values,
                    "y_pred": point_pred.reindex(y_test_series.index).values,
                    "interval_lower": lower.reindex(y_test_series.index).values,
                    "interval_upper": upper.reindex(y_test_series.index).values,
                }
            )
        )


# -------------------------------------------------------------------
# 2. Point grid-search candidates converted to interval forecasters
# -------------------------------------------------------------------

for i, params in enumerate(point_gscv.cv_results_["params"]):
    model_name = _model_label("point_conformal", i, params)

    point_forecaster = point_gscv.forecaster.clone()
    point_forecaster.set_params(**params)

    conformal_forecaster = ConformalIntervals(
        forecaster=point_forecaster,
        method="empirical",
        initial_window=30,
        n_jobs=-1,
    )

    conformal_forecaster.fit(
        y=y_train,
        X=X_train,
        fh=fh,
    )

    interval_pred = conformal_forecaster.predict_interval(
        fh=fh,
        X=X_test,
        coverage=coverage,
    )

    point_pred = _as_series(
        conformal_forecaster.predict(
            fh=fh,
            X=X_test,
        )
    )

    all_interval_predictions[model_name] = interval_pred

    overall_pinball = interval_pinball.evaluate(
        y_true=y_test,
        y_pred=interval_pred,
    )

    pinball_by_coverage = interval_pinball_by_coverage.evaluate(
        y_true=y_test,
        y_pred=interval_pred,
    )

    mape = point_mape.evaluate(
        y_true=y_test,
        y_pred=point_pred,
    )

    for cov in coverage:
        lower = interval_pred[_interval_column(interval_pred, cov, "lower")]
        upper = interval_pred[_interval_column(interval_pred, cov, "upper")]

        score_rows.append(
            {
                "model": model_name,
                "model_family": "point_conformal",
                "coverage": cov,
                "test_pinball_loss": overall_pinball,
                "test_pinball_loss_by_coverage": (
                    pinball_by_coverage.loc[cov]
                    if cov in getattr(pinball_by_coverage, "index", [])
                    else np.nan
                ),
                "test_mape": mape,
                "observed_coverage": (
                    (y_test_series >= lower) & (y_test_series <= upper)
                ).mean(),
                "average_interval_width": (upper - lower).mean(),
            }
        )

        plot_frames.append(
            pd.DataFrame(
                {
                    "timestamp": y_test_series.index,
                    "model": model_name,
                    "model_family": "point_conformal",
                    "coverage": cov,
                    "y_true": y_test_series.values,
                    "y_pred": point_pred.reindex(y_test_series.index).values,
                    "interval_lower": lower.reindex(y_test_series.index).values,
                    "interval_upper": upper.reindex(y_test_series.index).values,
                }
            )
        )


evaluation_scores = (
    pd.DataFrame(score_rows)
    .sort_values(["test_pinball_loss", "coverage", "average_interval_width"])
    .reset_index(drop=True)
)

plot_comparison_df = (
    pd.concat(plot_frames, ignore_index=True)
    .set_index("timestamp")
    .sort_index()
)

evaluation_scores, all_interval_predictions, plot_comparison_df

C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\sktime\utils\datetime.py:226: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x + by
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\sktime\utils\datetime.py:226: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x + by
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\sktime\utils\datetime.py:226: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x + by
C:\Users\valen\PycharmProjects\HtS_NDA\.venv\Lib\site-packages\sktime\utils\datetime.py:226: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x + by
C:\Users\valen\Pycha